In [18]:
from pathlib import Path
import duckdb
import numpy as np

print("1. Fetching unique customer list from 'paySim dataset.csv'...")

# Fetch all unique origin customers
unique_cust_df = duckdb.query("""
    SELECT DISTINCT nameOrig AS customer_id 
    FROM read_csv_auto('paySim dataset.csv')
""").df()

n_cust = len(unique_cust_df)
np.random.seed(42)

# Generate synthetic profile columns
unique_cust_df['account_type'] = np.random.choice(['SAVINGS', 'CHECKING', 'BUSINESS'], size=n_cust)
unique_cust_df['customer_segment'] = np.random.choice(['RETAIL', 'VIP', 'CORPORATE'], size=n_cust)
unique_cust_df['kyc_status'] = np.random.choice(['VERIFIED', 'PENDING'], size=n_cust, p=[0.85, 0.15])


unique_cust_df['risk_rating_base'] = np.random.choice(['LOW', 'MEDIUM', 'HIGH'], size=n_cust, p=[0.60, 0.30, 0.10])

# Save as CSV to Desktop
output_path = Path.home() / "Desktop" / "customer_profiles.csv"
unique_cust_df.to_csv(output_path, index=False)

print(f"-> Success: 'customer_profiles.csv' created for {n_cust:,} customers!")

1. Fetching unique customer list from 'paySim dataset.csv'...
-> Success: 'customer_profiles.csv' created for 6,353,307 customers!


In [19]:
import pandas as pd
import duckdb
import numpy as np

# 1. ADIM: Doğru Giriş ve Çıkış Tipleriyle SQL Sorgusu
sql_mule_query = """
SELECT 
    p.nameOrig AS customer_id,
    cp.account_type,
    cp.customer_segment,
    cp.kyc_status,
    cp.risk_rating_base,
    -- GİREN PARA (Inflow)
    SUM(CASE WHEN p.type = 'CASH_IN' THEN p.amount ELSE 0 END) AS total_inflow,
    -- ÇIKAN PARA (Outflow: Transfer, Cash Out, Payment)
    SUM(CASE WHEN p.type IN ('TRANSFER', 'CASH_OUT', 'PAYMENT') THEN p.amount ELSE 0 END) AS total_outflow,
    COUNT(p.amount) AS total_txn_count
FROM read_csv_auto('paySim dataset.csv') p
LEFT JOIN read_csv_auto('customer_profiles.csv') cp
       ON p.nameOrig = cp.customer_id
WHERE cp.customer_segment = 'RETAIL'
GROUP BY p.nameOrig, cp.account_type, cp.customer_segment, cp.kyc_status, cp.risk_rating_base
HAVING total_inflow >= 50000 OR total_outflow >= 50000
ORDER BY total_inflow DESC
"""

print("[1/2] DuckDB ile SQL JOIN sorgusu çalıştırılıyor...")
mule_df = duckdb.query(sql_mule_query).df()

# 2. ADIM: Pandas Katmanı (Drain Percentage ve Risk Skorlama)
print("[2/2] Pandas ile risk analizleri hesaplanıyor...")

# Drain Percentage (Boşaltma Oranı)
mule_df['drain_percentage'] = np.where(
    mule_df['total_inflow'] > 0,
    (mule_df['total_outflow'] / mule_df['total_inflow']) * 100,
    0
)

# Risk Seviyesi Atama
def calculate_risk(row):
    if row['total_inflow'] >= 200000 or row['kyc_status'] == 'PENDING':
        return 'CRITICAL'
    elif row['total_inflow'] >= 100000 or row['risk_rating_base'] == 'HIGH':
        return 'HIGH'
    else:
        return 'MEDIUM'

mule_df['final_risk_level'] = mule_df.apply(calculate_risk, axis=1)

# Filtreleme: Yüksek Riskliler
retail_high_risk_mules = mule_df[
    mule_df['final_risk_level'].isin(['HIGH', 'CRITICAL'])
].copy()

# Kaydetme
retail_high_risk_mules.to_csv('AML_Alerts_Money_Mule_Retail_HighRisk.csv', index=False)

print(f"-> Tamamlandı! {len(retail_high_risk_mules)} adet YÜKSEK RİSKLİ alarm süzüldü.")

retail_high_risk_mules.head(10)

[1/2] DuckDB ile SQL JOIN sorgusu çalıştırılıyor...
[2/2] Pandas ile risk analizleri hesaplanıyor...
-> Tamamlandı! 508071 adet YÜKSEK RİSKLİ alarm süzüldü.


,customer_id,account_type,customer_segment,kyc_status,risk_rating_base,total_inflow,total_outflow,total_txn_count,drain_percentage,final_risk_level
0,C1072983741,CHECKING,RETAIL,PENDING,LOW,1821279.53,0.0,1,0.0,CRITICAL
1,C1719006755,CHECKING,RETAIL,PENDING,LOW,1782621.49,0.0,1,0.0,CRITICAL
2,C1139859398,SAVINGS,RETAIL,PENDING,MEDIUM,1675137.60,0.0,1,0.0,CRITICAL
3,C1065575100,CHECKING,RETAIL,VERIFIED,LOW,1602113.20,0.0,1,0.0,CRITICAL
4,C263860433,BUSINESS,RETAIL,VERIFIED,MEDIUM,1529738.97,0.0,1,0.0,CRITICAL
5,C1095502692,CHECKING,RETAIL,VERIFIED,LOW,1497912.42,0.0,1,0.0,CRITICAL
6,C1278066124,BUSINESS,RETAIL,VERIFIED,HIGH,1464307.42,0.0,1,0.0,CRITICAL
7,C1276960686,CHECKING,RETAIL,VERIFIED,LOW,1409644.73,0.0,1,0.0,CRITICAL
8,C2015800554,SAVINGS,RETAIL,VERIFIED,LOW,1394689.81,0.0,1,0.0,CRITICAL
9,C991450871,CHECKING,RETAIL,VERIFIED,MEDIUM,1380810.22,0.0,1,0.0,CRITICAL
